In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass


EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.image-classify.detect-vandalism",
        description="Classifies whether the infrastructure in the image is clean or vandalized.",
        worker_release="qwen3-instruct",
        text_prompt="""
          Determine the primary content of the image and assign exactly one label: ['Pass_clean', 'Fail_vandalized'].
          Focus only on the single main piece of infrastructure or public/commercial property that is the clear subject of the image, such as a wall, shutter, sign, bench, transit vehicle, or utility box. Ignore background items, unrelated objects, people, and vehicles that are not the main subject.
          Choose 'Pass_clean' only if the image's main focus shows no unauthorized markings, damage, or defacement. The surface should appear as originally intended — intact, unmarked, and free of graffiti, stickers, posters, scratches, dents, or breakage.
          Choose 'Fail_vandalized' if the image's main focus shows any unauthorized alteration or damage, including but not limited to:
          Spray paint, tagging, markers, or etched graffiti
          Scratched, keyed, or gouged surfaces without paint
          Broken, smashed, or shattered glass
          Dents, kicks, or bent/broken structural elements (signage, fencing, poles, fixtures)
          Unauthorized stickers or wheatpasted posters/flyers
          Scorch or burn marks
          Forced-open panels or broken locks
          Normal wear such as faded paint, dirt from weather exposure, or minor rust should not be classified as vandalized unless it is accompanied by one of the issues above.
          If multiple types of damage are visible, still return 'Fail_vandalized' — this label does not distinguish between vandalism subtypes.
          Return only the single best-fitting label.

        """,
        transform_into=TransformInto(),
       config=InferRuntimeConfig(
           max_new_tokens=5,
           image_size=640
        ),
        is_public=False
    )
]



### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Image


In [ ]:
from pathlib import Path


pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.image-classify.detect-vandalism:latest"
   )
])


with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_img_path = Path("/content/sample_img.jpg") # change to the path of your sample image
   job = endpoint.upload(sample_img_path)
   while result := job.predict():
     print(json.dumps(result, indent=2))

print("Done")
